<a href="https://colab.research.google.com/github/tharujayasinghe163/Statistical-Learning-e22163/blob/main/Assignment_7c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



## Part 1: Bayesian Estimation of a User Ability Parameter from Item Responses

### Task 1: Mechanics of the 2PL Item Response Curve

In the Two-Parameter Logistic (2PL) model, the probability of a correct answer given latent ability $\Theta = \theta$ is:


$$p_i(\theta) = P(Y_i = 1 \mid \Theta = \theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$

* **Difficulty Parameter ($b_i$):** Represents the location on the ability scale where the probability of answering correctly is exactly $0.5$ ($p_i(b_i) = 0.5$). Increasing $b_i$ shifts the logistic curve horizontally to the right, meaning a higher latent ability $\theta$ is required to achieve the same probability of success.
* **Discrimination Parameter ($a_i$):** Controls the steepness/slope of the curve at $\theta = b_i$. Higher values of $a_i$ make the transition sharper, allowing the item to discriminate effectively between abilities just above or below $b_i$.

```python
import numpy as np
import plotly.graph_objects as go

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta_vals = np.linspace(-6, 6, 300)
curves = [
    {"a": 0.5, "b": 0, "style": "dash"},
    {"a": 1.5, "b": -2, "style": "solid"},
    {"a": 1.5, "b": 0, "style": "solid"},
    {"a": 1.5, "b": 2, "style": "solid"},
]

fig = go.Figure()
for c in curves:
    fig.add_trace(go.Scatter(
        x=theta_vals, y=p_i(theta_vals, c["a"], c["b"]),
        mode='lines', name=f"a = {c['a']}, b = {c['b']}",
        line=dict(dash=c["style"], width=2.5)
    ))

fig.update_layout(
    title="2PL Item Response Curves",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i = 1 | θ)",
    template="plotly_white"
)
fig.show()

```

---

### Task 2: Sequential Likelihood Contribution & Joint Likelihood

For a **single response** $y_k \in \{0, 1\}$ at step $k$:


$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

Assuming conditional independence of item responses given $\theta$, the **joint likelihood function** for the running response history vector $y^{(k)} = (y_1, y_2, \dots, y_k)$ is:


$$L(y^{(k)} \mid \theta) = \prod_{i=1}^k [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

---

### Task 3: Mathematical Formulation of the Running Update

Using the posterior from step $k-1$ as the prior for step $k$, the normalized recursive posterior density is:


$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) = \frac{[p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} \, f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})}{\int_{-\infty}^{\infty} [p_k(s)]^{y_k} [1 - p_k(s)]^{1 - y_k} \, f_{\Theta \mid Y^{(k-1)}}(s \mid y^{(k-1)}) \, ds}$$

Expressed up to a **proportionality constant**:


$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$

---

### Task 4: Dynamic Shifting

When a user correctly answers ($y_k = 1$) a highly difficult item (large $b_k$), the likelihood function $L(y_k \mid \theta) = p_k(\theta)$ is close to zero for low ability levels ($\theta < b_k$) and increases monotonically toward 1 for higher ability levels ($\theta > b_k$).

Multiplying the step $(k-1)$ prior by this increasing likelihood penalizes the probability density in lower ability regions while retaining mass in higher ability regions. This causes the mode (MAP) and mean (Bayes estimate) of the running posterior density curve to **shift significantly to the right**.

---

### Task 5: Tracking Certainty and Sharpness

* **Large Discrimination ($a_k \gg 0$):** The response function $p_k(\theta)$ approaches a step function at $\theta = b_k$. This sharp multiplier heavily truncates density on one side of $b_k$, causing a steep drop in posterior variance and significantly **increasing the sharpness/certainty** of the updated distribution.
* **Small Discrimination ($a_k \approx 0$):** The response function $p_k(\theta) \approx 0.5$ across all $\theta$. The likelihood is essentially flat, contributing almost no new information, leaving the posterior variance and shape virtually unchanged.

---

### Task 6: Numerical Implementation of a Running Grid

1. **Grid Discretization:** Define a fixed grid of $M$ points $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$ spanning a bounded range (e.g., $[-5, 5]$).
2. **Prior Evaluation:** Initialize the density vector on the grid using the standard normal distribution:

$$\mathbf{f}^{(0)} = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\boldsymbol{\theta}^2}{2}\right)$$


3. **Likelihood Computation:** At step $k$, compute likelihood values across the grid:

$$\mathbf{L}_k = [p_k(\boldsymbol{\theta})]^{y_k} \odot [1 - p_k(\boldsymbol{\theta})]^{1 - y_k}$$


4. **Unnormalized Update:** Compute element-wise product:

$$\mathbf{q}^{(k)} = \mathbf{L}_k \odot \mathbf{f}^{(k-1)}$$


5. **Numerical Normalization:** Compute integral $I_k$ using the trapezoidal rule and scale the density:

$$I_k = \text{trapezoid}(\mathbf{q}^{(k)}, \boldsymbol{\theta}), \quad \mathbf{f}^{(k)} = \frac{\mathbf{q}^{(k)}}{I_k}$$



---

### Task 7: Evaluating Convergence over the Timeline (Python Implementation)

```python
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# 1. Setup Simulation Parameters
np.random.seed(42)
theta_true = 0.75
n_items = 20
grid_size = 500
theta_grid = np.linspace(-5, 5, grid_size)

# 2. Base Prior Initialization: N(0, 1)
current_posterior = stats.norm.pdf(theta_grid, 0, 1)
current_posterior /= np.trapezoid(current_posterior, theta_grid)

bayes_estimates = [np.trapezoid(theta_grid * current_posterior, theta_grid)]
map_estimates = [theta_grid[np.argmax(current_posterior)]]

# 3. Running Sequential Loop
for k in range(1, n_items + 1):
    a_k = np.random.uniform(0.5, 2.0)
    b_k = np.random.normal(0.0, 1.0)
    
    # Simulate user response against true ability
    p_true = 1 / (1 + np.exp(-a_k * (theta_true - b_k)))
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0
    
    # Compute likelihood across grid
    p_grid = 1 / (1 + np.exp(-a_k * (theta_grid - b_k)))
    likelihood = (p_grid ** y_k) * ((1 - p_grid) ** (1 - y_k))
    
    # Posterior update and normalization
    current_posterior *= likelihood
    current_posterior /= np.trapezoid(current_posterior, theta_grid)
    
    # Calculate point estimates
    theta_bayes = np.trapezoid(theta_grid * current_posterior, theta_grid)
    theta_map = theta_grid[np.argmax(current_posterior)]
    
    bayes_estimates.append(theta_bayes)
    map_estimates.append(theta_map)

# 4. Visualization
steps = list(range(n_items + 1))
fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=bayes_estimates, mode='lines+markers', name='Posterior Mean (Bayes)'))
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name='MAP Estimate'))
fig.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True θ = 0.75")

fig.update_layout(
    title="Sequential Convergence of Latent Ability Estimators (2PL IRT)",
    xaxis_title="Item Step (k)",
    yaxis_title="Estimated Latent Ability (θ)",
    template="plotly_white"
)
fig.show()

```

#### Analysis & Interpretation

As item count $k$ increases, accumulated item observations narrow the variance of the running posterior distribution. Consequently, the distance between both estimators ($\hat{\theta}_{\text{Bayes}}^{(k)}$ and $\hat{\theta}_{\text{MAP}}^{(k)}$) and the true value $\theta_{\text{true}} = 0.75$ shrinks towards zero. This diminishing gap reflects higher confidence and statistical precision in measuring the user's true ability parameter.

---

---

## Part 2: Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

### Model Setup

Let $\theta \in (0, 1)$ denote the underlying Click-Through Rate (CTR).

* **Binomial Likelihood:** For a batch of $n_k$ impressions yielding $y_k$ clicks:

$$P(Y_k = y_k \mid \theta) = \binom{n_k}{y_k} \theta^{y_k} (1 - \theta)^{n_k - y_k}$$


* **Conjugate Beta Prior:**

$$f(\theta; \alpha_{k-1}, \beta_{k-1}) = \frac{1}{\text{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1}$$



### Exact Closed-Form Analytic Update

Combining the prior and likelihood yields:


$$f(\theta \mid y_k) \propto \left[ \theta^{y_k} (1 - \theta)^{n_k - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right] = \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + n_k - y_k) - 1}$$

Because Beta is conjugate to Binomial, the updated posterior parameters at step $k$ are computed algebraically:


$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (n_k - y_k)$$

### Key Summary Metrics

* **Posterior Mean (Bayes Estimate):** $\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$
* **Posterior Mode (MAP):** $\hat{\theta}_{\text{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad (\text{for } \alpha_k, \beta_k > 1)$
* **Posterior Variance:** $\text{Var}(\theta \mid \text{data}) = \frac{\alpha_k \beta_k}{(\alpha_k + \beta_k)^2 (\alpha_k + \beta_k + 1)}$

---

### Python Real-Time CTR Tracking Implementation

```python
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Parameters
true_ctr = 0.045  # 4.5% CTR
n_batches = 15
impressions_per_batch = 1000

# Initialize Prior: Beta(a=2, b=98) -> Prior Mean ~2%
a_curr, b_curr = 2, 98

means, low_ci, high_ci = [], [], []

for batch in range(1, n_batches + 1):
    # Generate clicks from Binomial process
    clicks = np.random.binomial(impressions_per_batch, true_ctr)
    non_clicks = impressions_per_batch - clicks
    
    # Closed-form Beta parameter update
    a_curr += clicks
    b_curr += non_clicks
    
    # Compute metrics
    post_mean = a_curr / (a_curr + b_curr)
    ci_lower = stats.beta.ppf(0.025, a_curr, b_curr)
    ci_upper = stats.beta.ppf(0.975, a_curr, b_curr)
    
    means.append(post_mean)
    low_ci.append(ci_lower)
    high_ci.append(ci_upper)

# Plotting with Plotly
batches = list(range(1, n_batches + 1))
fig = go.Figure()

# 95% Credible Interval Band
fig.add_trace(go.Scatter(
    x=batches + batches[::-1],
    y=high_ci + low_ci[::-1],
    fill='toself',
    fillcolor='rgba(0,100,80,0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='95% Credible Interval'
))

# Posterior Mean
fig.add_trace(go.Scatter(
    x=batches, y=means,
    mode='lines+markers',
    line=dict(color='rgb(0,100,80)'),
    name='Posterior Mean CTR'
))

# Target CTR
fig.add_hline(y=true_ctr, line_dash="dash", line_color="red", annotation_text="True CTR = 4.5%")

fig.update_layout(
    title="Real-Time Bayesian CTR Tracking (Beta-Binomial Updates)",
    xaxis_title="Batch Number",
    yaxis_title="CTR",
    template="plotly_white"
)
fig.show()

```